In [ ]:
%sql
-- Filas por mes: ¿falta alguno? ¿alguno tiene muy pocas?
SELECT archivo_origen, COUNT(*) AS filas
FROM workspace.movilidad.viajes_stm
GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
WITH cargados AS (
  SELECT DISTINCT
    CASE WHEN left(archivo_origen, 2) = '20' THEN archivo_origen
         ELSE concat(right(archivo_origen, 4), left(archivo_origen, 2)) END AS mes
  FROM workspace.movilidad.viajes_stm
),
esperados AS (
  SELECT date_format(m, 'yyyyMM') AS mes
  FROM (SELECT explode(sequence(DATE'2020-01-01', DATE'2026-09-01', INTERVAL 1 MONTH)) AS m)
)
SELECT e.mes AS mes_faltante
FROM esperados e LEFT JOIN cargados c ON e.mes = c.mes
WHERE c.mes IS NULL
ORDER BY 1;

In [ ]:
%sql
-- Nulos y duplicados
SELECT COUNT(*) AS total,
       COUNT(DISTINCT id_viaje) AS ids_distintos,
       SUM(CASE WHEN id_viaje IS NULL THEN 1 ELSE 0 END) AS id_nulos,
       SUM(CASE WHEN fecha_evento IS NULL THEN 1 ELSE 0 END) AS fecha_nulas
FROM workspace.movilidad.viajes_stm;

In [ ]:
%sql
-- Categorías: ¿valores raros o mal escritos?
SELECT grupo_usuario, tipo_viaje, COUNT(*) AS filas
FROM workspace.movilidad.viajes_stm
GROUP BY 1, 2 ORDER BY 3 DESC;

In [ ]:
%sql
   SELECT COUNT(*) - COUNT(DISTINCT concat_ws('|', *)) AS filas_repetidas
   FROM workspace.movilidad.viajes_stm
   WHERE archivo_origen = '092026';

In [ ]:
%sql
   SELECT COUNT(DISTINCT archivo_origen) FROM workspace.movilidad.viajes_stm;

In [ ]:
%sql
SELECT DISTINCT descripcion_tipo_viaje
FROM workspace.movilidad.viajes_stm
LIMIT 20;

In [ ]:
%sql
SELECT * FROM workspace.movilidad.viajes_stm LIMIT 10;

In [ ]:
%sql
WITH base AS (
  SELECT
    CASE WHEN left(archivo_origen, 2) = '20' THEN archivo_origen
         ELSE concat(right(archivo_origen, 4), left(archivo_origen, 2)) END AS mes,
    COUNT(*) AS filas,
    COUNT(DISTINCT left(fecha_evento, 10)) AS dias_con_datos,
    MIN(fecha_evento) AS primera,
    MAX(fecha_evento) AS ultima
  FROM workspace.movilidad.viajes_stm
  GROUP BY 1
)
SELECT mes, filas, dias_con_datos,
       day(last_day(to_date(concat(mes, '01'), 'yyyyMMdd'))) AS dias_del_mes,
       primera, ultima,
       CASE WHEN dias_con_datos < day(last_day(to_date(concat(mes, '01'), 'yyyyMMdd')))
            THEN 'FALTAN DIAS'
            WHEN filas < 0.5 * AVG(filas) OVER () THEN 'POCAS FILAS'
            ELSE 'OK' END AS estado
FROM base
ORDER BY filas;